# menpofit visualisations

Trains tiny models of six greyscale images (about 15 seconds), so the shapes are not meant to be good: this is for checking the widgets. Needs `menpowidgets[menpofit]`.

**Try:** parameters and Play on the AAM/CLM, the *Shape* / *Appearance* sub-tabs, Variance, the *Experts* tab (patches range `0:5`, offset, background toggle, bounding boxes), and in the fitting results the *Final* / *Iterations* / *Costs* / *Errors* buttons, figure mode Single/Multiple.

In [ ]:
%matplotlib inline
import warnings; warnings.simplefilter("ignore")
import numpy as np
import menpo.io as mio
from menpo.model import PCAModel
from menpo.shape import PointCloud
import menpowidgets as mw
print("menpowidgets", mw.__version__)

In [ ]:
from menpofit import aam, atm, clm
GROUP = "PTS"
g = mio.import_builtin_asset.takeo_ppm().as_greyscale()
train = [g.rotate_ccw_about_centre(a).rescale(0.5) for a in (-6, -3, 0, 3, 6, 9)]
shapes = [i.landmarks[GROUP] for i in train]
COMMON = dict(group=GROUP, scales=(1.0,), diagonal=50, verbose=False)
holistic_aam = aam.HolisticAAM(train, max_shape_components=3, max_appearance_components=3, **COMMON)
patch_aam = aam.PatchAAM(train, max_shape_components=3, max_appearance_components=3, patch_shape=(5, 5), **COMMON)
holistic_atm = atm.HolisticATM(train[0], shapes, max_shape_components=3, **COMMON)
patch_atm = atm.PatchATM(train[0], shapes, max_shape_components=3, patch_shape=(5, 5), **COMMON)
the_clm = clm.CLM(train, max_shape_components=3, patch_shape=(5, 5), **COMMON)
fitter = aam.LucasKanadeAAMFitter(holistic_aam, n_shape=3, n_appearance=3)
results = [fitter.fit_from_bb(im, im.landmarks[GROUP].bounding_box(), max_iters=4, gt_shape=im.landmarks[GROUP]) for im in train[:3]]
print("ready")

## AAMs

In [ ]:
mw.visualize_aam(holistic_aam)

In [ ]:
mw.visualize_aam(holistic_aam, mode='single')

In [ ]:
mw.visualize_patch_aam(patch_aam)

## ATMs

In [ ]:
mw.visualize_atm(holistic_atm)

In [ ]:
mw.visualize_patch_atm(patch_atm)

## CLM and its experts

In [ ]:
mw.visualize_clm(the_clm)

In [ ]:
mw.visualize_expert_ensemble(the_clm.expert_ensembles[0], the_clm.reference_shape)

## Fitting results and CED curves

In [ ]:
mw.visualize_fitting_results(results)

In [ ]:
mw.plot_ced([[0.01 * k * (i + 1) for k in range(1, 30)] for i in range(2)], legend_entries=["method a", "method b"])